In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv('Obesity Classification.csv')

In [3]:
df.shape

(108, 7)

In [4]:
df.head()

,ID,Age,Gender,Height,Weight,BMI,Label
0,1,25,Male,175,80,25.3,Normal Weight
1,2,30,Female,160,60,22.5,Normal Weight
2,3,35,Male,180,90,27.3,Overweight
3,4,40,Female,150,50,20.0,Underweight
4,5,45,Male,190,100,31.2,Obese


Remove Null Values

In [5]:
df.dropna()

,ID,Age,Gender,Height,Weight,BMI,Label
0,1,25,Male,175,80,25.3,Normal Weight
1,2,30,Female,160,60,22.5,Normal Weight
2,3,35,Male,180,90,27.3,Overweight
3,4,40,Female,150,50,20.0,Underweight
4,5,45,Male,190,100,31.2,Obese
...,...,...,...,...,...,...,...
103,106,11,Male,175,10,3.9,Underweight
104,107,16,Female,160,10,3.9,Underweight
105,108,21,Male,180,15,5.6,Underweight
106,109,26,Female,150,15,5.6,Underweight


In [6]:
df.shape

(108, 7)

In [7]:
df.dtypes

ID          int64
Age         int64
Gender     object
Height      int64
Weight      int64
BMI       float64
Label      object
dtype: object

Label Encode Categorical Data

In [8]:
from sklearn.preprocessing import LabelEncoder
le= LabelEncoder()

df['Gender'] = le.fit_transform(df['Gender'])
df.head()

,ID,Age,Gender,Height,Weight,BMI,Label
0,1,25,1,175,80,25.3,Normal Weight
1,2,30,0,160,60,22.5,Normal Weight
2,3,35,1,180,90,27.3,Overweight
3,4,40,0,150,50,20.0,Underweight
4,5,45,1,190,100,31.2,Obese


In [9]:
from sklearn.preprocessing import LabelEncoder
le= LabelEncoder()

df['Label'] = le.fit_transform(df['Label'])
df.head()

,ID,Age,Gender,Height,Weight,BMI,Label
0,1,25,1,175,80,25.3,0
1,2,30,0,160,60,22.5,0
2,3,35,1,180,90,27.3,2
3,4,40,0,150,50,20.0,3
4,5,45,1,190,100,31.2,1


In [10]:
df.drop(labels='ID',axis=1,inplace=True)
df.head()

,Age,Gender,Height,Weight,BMI,Label
0,25,1,175,80,25.3,0
1,30,0,160,60,22.5,0
2,35,1,180,90,27.3,2
3,40,0,150,50,20.0,3
4,45,1,190,100,31.2,1


Train Test Split

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop('Label', axis=1)
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [164]:
X_train.head()

,Age,Gender,Height,Weight,BMI
64,46,1,200,85,26.1
26,98,1,200,110,34.2
22,78,1,180,90,27.3
31,29,1,180,85,26.1
47,62,0,120,85,27.5


In [167]:
y_train.head()

64    2
26    1
22    2
31    2
47    2
Name: Label, dtype: int32

In [12]:
X_train.corr()

,Age,Gender,Height,Weight,BMI
Age,1.000000,-0.126646,-0.119888,0.441991,0.460286
Gender,-0.126646,1.000000,0.887768,0.397455,0.319869
Height,-0.119888,0.887768,1.000000,0.409562,0.338660
Weight,0.441991,0.397455,0.409562,1.000000,0.972020
BMI,0.460286,0.319869,0.338660,0.972020,1.000000


Scaling the Data

In [129]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Applying PCA

In [15]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [16]:
X_train_pca.shape

(86, 3)

In [17]:
X_test_pca.shape

(22, 3)

In [18]:
X_train_pca = pd.DataFrame(X_train_pca)
X_test_pca=pd.DataFrame(X_test_pca)

In [19]:
X_train_pca.head()

,0,1,2
0,0.687487,-0.078162,-0.074445
1,0.772336,-0.632883,0.170234
2,0.589845,-0.324898,0.153502
3,0.583302,-0.003602,-0.213356
4,-0.563545,-0.483498,-0.170942


Variance explained by features

In [20]:
print(pca.explained_variance_ratio_)

[0.67629486 0.23068771 0.0614363 ]


In [23]:
cumulative_explained_variance = np.cumsum(pca.explained_variance_ratio_)
cumulative_explained_variance

array([0.67629486, 0.90698257, 0.96841887])

KNN Classifier

In [24]:
from sklearn.neighbors import KNeighborsClassifier
model1 = KNeighborsClassifier()

In [25]:
y_train=y_train.astype('int')
y_test=y_test.astype('int')

In [26]:
import timeit
training_time = timeit.timeit(lambda: model1.fit(X_train_pca, y_train), number=1) 

In [28]:
start_time = timeit.default_timer()
y_pred = model1.predict(X_test_pca)
testing_time = timeit.default_timer() - start_time

In [29]:
print("Average Training Time:", training_time) 
print("Testing Time:", testing_time)

Average Training Time: 0.010239799972623587
Testing Time: 0.013932699977885932


In [34]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
confusion_matrix(y_test, y_pred)

array([[5, 0, 1, 0],
       [0, 4, 0, 0],
       [1, 0, 3, 0],
       [1, 0, 0, 7]], dtype=int64)

In [31]:
print("Accuracy: ")
accuracy_score(y_test, y_pred)

Accuracy: 


0.8636363636363636

In [32]:
print("Precision: ")
precision_score(y_test, y_pred, average='weighted')

Precision: 


0.8766233766233765

In [33]:
print("Recall: ")
recall_score(y_test, y_pred, average='weighted')

Recall: 


0.8636363636363636

In [48]:
print("F1 Score: ")
f1_score(y_test, y_pred, average='weighted')

F1 Score: 


0.8673659673659674

Logistic Regression

In [50]:
from sklearn.linear_model import LogisticRegression
model2 = LogisticRegression()

In [51]:
training_time = timeit.timeit(lambda: model2.fit(X_train_pca, y_train), number=1) 

In [52]:
start_time = timeit.default_timer()
y_pred = model2.predict(X_test_pca)
testing_time = timeit.default_timer() - start_time

In [53]:
print("Average Training Time:", training_time) 
print("Testing Time:", testing_time)

Average Training Time: 0.0223202999914065
Testing Time: 0.006410299974959344


In [54]:
confusion_matrix(y_test, y_pred)

array([[4, 0, 0, 2],
       [0, 1, 3, 0],
       [4, 0, 0, 0],
       [0, 0, 0, 8]], dtype=int64)

In [55]:
print("Accuracy: ")
accuracy_score(y_test, y_pred)

Accuracy: 


0.5909090909090909

In [56]:
print("Precision: ")
precision_score(y_test, y_pred, average='weighted')

Precision: 


0.6090909090909091

In [57]:
print("Recall: ")
recall_score(y_test, y_pred, average='weighted')

Recall: 


0.5909090909090909

In [58]:
print("F1 Score: ")
f1_score(y_test, y_pred, average='weighted')

F1 Score: 


0.5518037518037517

Random Forest

In [59]:
from sklearn.ensemble import RandomForestClassifier
model3 = RandomForestClassifier()

In [65]:
training_time = timeit.timeit(lambda: model3.fit(X_train_pca, y_train), number=1) 

In [66]:
start_time = timeit.default_timer()
y_pred = model3.predict(X_test_pca)
testing_time = timeit.default_timer() - start_time

In [67]:
print("Average Training Time:", training_time) 
print("Testing Time:", testing_time)

Average Training Time: 0.19579279999015853
Testing Time: 0.03707530000247061


In [68]:
confusion_matrix(y_test, y_pred)

array([[5, 0, 1, 0],
       [0, 3, 1, 0],
       [1, 0, 3, 0],
       [1, 0, 0, 7]], dtype=int64)

In [69]:
print("Accuracy: ")
accuracy_score(y_test, y_pred)

Accuracy: 


0.8181818181818182

In [70]:
print("Precision: ")
precision_score(y_test, y_pred, average='weighted')

Precision: 


0.8493506493506492

In [71]:
print("Recall: ")
recall_score(y_test, y_pred, average='weighted')

Recall: 


0.8181818181818182

In [72]:
print("F1 Score: ")
f1_score(y_test, y_pred, average='weighted')

F1 Score: 


0.8262404262404263

Without PCA

Random Forest

In [104]:
training_time = timeit.timeit(lambda: model3.fit(X_train_scaled, y_train), number=1) 

In [105]:
start_time = timeit.default_timer()
y_pred = model3.predict(X_test_scaled)
testing_time = timeit.default_timer() - start_time

In [106]:
print("Average Training Time:", training_time) 
print("Testing Time:", testing_time)

Average Training Time: 0.17048820003401488
Testing Time: 0.014915100007783622


In [107]:
confusion_matrix(y_test, y_pred)

array([[6, 0, 0, 0],
       [0, 4, 0, 0],
       [0, 0, 4, 0],
       [0, 0, 0, 8]], dtype=int64)

In [108]:
print("Accuracy: ")
accuracy_score(y_test, y_pred)

Accuracy: 


1.0

In [109]:
print("Precision: ")
precision_score(y_test, y_pred, average='weighted')

Precision: 


1.0

In [110]:
print("Recall: ")
recall_score(y_test, y_pred, average='weighted')

Recall: 


1.0

In [111]:
print("F1 Score: ")
f1_score(y_test, y_pred, average='weighted')

F1 Score: 


1.0

Logistic Regression

In [112]:
training_time = timeit.timeit(lambda: model2.fit(X_train_scaled, y_train), number=1) 

In [113]:
start_time = timeit.default_timer()
y_pred = model2.predict(X_test_scaled)
testing_time = timeit.default_timer() - start_time

In [114]:
print("Average Training Time:", training_time) 
print("Testing Time:", testing_time)

Average Training Time: 0.009350000007543713
Testing Time: 0.0005074000218883157


In [115]:
confusion_matrix(y_test, y_pred)

array([[4, 0, 0, 2],
       [0, 3, 1, 0],
       [4, 0, 0, 0],
       [0, 0, 0, 8]], dtype=int64)

In [116]:
print("Accuracy: ")
accuracy_score(y_test, y_pred)

Accuracy: 


0.6818181818181818

In [117]:
print("Precision: ")
precision_score(y_test, y_pred, average='weighted')

Precision: 


0.6090909090909091

In [118]:
print("Recall: ")
recall_score(y_test, y_pred, average='weighted')

Recall: 


0.6818181818181818

In [119]:
print("F1 Score: ")
f1_score(y_test, y_pred, average='weighted')

F1 Score: 


0.6349206349206349

KNN

In [120]:
training_time = timeit.timeit(lambda: model1.fit(X_train_scaled, y_train), number=1) 

In [121]:
start_time = timeit.default_timer()
y_pred = model1.predict(X_test_scaled)
testing_time = timeit.default_timer() - start_time

In [122]:
print("Average Training Time:", training_time) 
print("Testing Time:", testing_time)

Average Training Time: 0.00118090002797544
Testing Time: 0.0030288000125437975


In [123]:
confusion_matrix(y_test, y_pred)

array([[5, 0, 1, 0],
       [0, 3, 1, 0],
       [1, 0, 3, 0],
       [0, 0, 0, 8]], dtype=int64)

In [124]:
print("Accuracy: ")
accuracy_score(y_test, y_pred)

Accuracy: 


0.8636363636363636

In [125]:
print("Precision: ")
precision_score(y_test, y_pred, average='weighted')

Precision: 


0.8818181818181817

In [126]:
print("Recall: ")
recall_score(y_test, y_pred, average='weighted')

Recall: 


0.8636363636363636

In [127]:
print("F1 Score: ")
f1_score(y_test, y_pred, average='weighted')

F1 Score: 


0.8679653679653679

Custom Testing

In [146]:
X_train.head()

,Age,Gender,Height,Weight,BMI
64,46,1,200,85,26.1
26,98,1,200,110,34.2
22,78,1,180,90,27.3
31,29,1,180,85,26.1
47,62,0,120,85,27.5


In [157]:
columns = X_train.columns
df1 = pd.DataFrame(columns=columns)

In [158]:
AR = {'Age': 20,
        'Gender': 1,
        'Height': 178,
        'Weight': 72,
        'BMI': 22.7}

NG={'Age': 20,
        'Gender': 0,
        'Height': 165,
        'Weight': 54,
        'BMI': 19.8}

In [159]:
df1=df1.append(AR, ignore_index=True)
df1=df1.append(NG, ignore_index=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_19484\3679951590.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df1=df1.append(AR, ignore_index=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_19484\3679951590.py:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df1=df1.append(NG, ignore_index=True)


In [160]:
df1.head()

,Age,Gender,Height,Weight,BMI
0,20.0,1.0,178.0,72.0,22.7
1,20.0,0.0,165.0,54.0,19.8


In [161]:
df1_scaled=scaler.transform(df1)

In [162]:
pred=model3.predict(df1_scaled)

In [163]:
pred

array([0, 3])